# EEG Model Optimization - Colab

**Optimized for speed and minimal RAM usage:**
- Always uses balanced class weights (no optimization needed)
- Aggressive memory cleanup after each model
- Fast parameter search with proven parameter ranges

**Models included:** EEGNet, CBraMod, CSP, Feature-based (RF/SVM), Eye-tracking (RF/SVM)

In [ ]:
# Mount Google Drive and install packages
from google.colab import drive
drive.mount('/content/drive')

# Install required packages (optimized list)
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q mne einops  # einops needed for CBraMod

# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU available - models will run on CPU (slower)")

In [ ]:
# Navigate to repository and configure paths
import os
import sys
from pathlib import Path

# Update this path to match your Drive setup
%cd /content/drive/MyDrive/eye-eeg/eye-eeg
print(f"Repository directory: {os.getcwd()}")

# Path configuration for Colab
COLAB_REPO_PATH = "/content/drive/MyDrive/eye-eeg/eye-eeg"
COLAB_DATA_PATH = "/content/drive/MyDrive/eye-eeg"

# Add to path
sys.path.insert(0, COLAB_REPO_PATH)

# Patch data path for Colab environment
def colab_get_data_path():
    return Path(COLAB_DATA_PATH)

# Import and patch optimization module
from classification import model_optimization_runner
model_optimization_runner.get_data_path = colab_get_data_path

print(f"✅ Path configuration updated for Colab")
print(f"Repository: {COLAB_REPO_PATH}")
print(f"Data path: {COLAB_DATA_PATH}")

In [ ]:
# Import optimization functions
from classification.model_optimization_runner import (
    optimize_all_models, test_all_models, optimize_single_model,
    get_data_path, get_results_path, PARAM_GRIDS, aggressive_cleanup
)
from data.data_configs import SUBJECTS

print("✅ Optimization functions imported successfully")
print(f"Data path: {get_data_path()}")
print(f"Results path: {get_results_path()}")

print(f"\nAvailable models:")
for model_type, params in PARAM_GRIDS.items():
    param_combos = 1
    for param_values in params.values():
        param_combos *= len(param_values)
    print(f"  {model_type}: {param_combos} parameter combinations")

print(f"\nAvailable subjects: {len(SUBJECTS)} total")
print(f"First few: {SUBJECTS[:5]}")

## Step 1: Test Pipeline (No Training)

Verify that models can be created and data loaded correctly before starting optimization.

In [ ]:
# Test the pipeline first (no training, just verification)
print("=== TESTING PIPELINE (No Training) ===")
print("This will verify models can be created and data loaded correctly")

# Test with minimal subjects and test mode
test_results = test_all_models()

print("\n✅ Pipeline test completed!")
print("If this works, you can proceed to actual optimization below.")

## Step 2: Quick Optimization (Recommended)

Fast optimization with subset of subjects for testing. Takes about 15-30 minutes.

In [ ]:
# Quick optimization (recommended for first run)
print("=== QUICK OPTIMIZATION ===")
print("Training with limited subjects and parameters")
print("⏱️  Expected time: 15-30 minutes")

# Use subjects that have data available (adjust based on your data)
optimization_subjects = SUBJECTS[:5]  # Adjust this based on your available data

quick_results = optimize_all_models(
    subjects=optimization_subjects,
    max_trials=15,  # Reasonable for quick testing
    split_strategies=['normal']  # Just normal split for speed
)

print("\n✅ Quick optimization completed!")
print("Results saved to classification/results/")

## Step 3: Results Analysis

Load and analyze the optimization results.

In [ ]:
# Load and display optimization results
import json
import pandas as pd
import matplotlib.pyplot as plt

results_path = get_results_path()
print(f"Looking for results in: {results_path}")

# Load latest optimization results
split_strategy = 'normal'
param_file = results_path / f"optimal_parameters_{split_strategy}_latest.json"

if param_file.exists():
    with open(param_file, 'r') as f:
        optimal_params = json.load(f)
    
    print(f"\n✅ Loaded optimal parameters from {param_file.name}")
    
    # Show optimal parameters summary
    print("\n=== OPTIMIZATION RESULTS ===")
    results_data = []
    for model_name, info in optimal_params.items():
        results_data.append({
            'Model': model_name,
            'Validation Score': info['score'],
            'Parameters': str(info['params'])[:50] + "..." if len(str(info['params'])) > 50 else str(info['params'])
        })
    
    df_optimization = pd.DataFrame(results_data)
    df_optimization = df_optimization.sort_values('Validation Score', ascending=False)
    print(df_optimization)
    
    # Plot validation scores
    plt.figure(figsize=(12, 6))
    plt.bar(df_optimization['Model'], df_optimization['Validation Score'], alpha=0.7)
    plt.title('Model Optimization Results - Validation Scores')
    plt.ylabel('Balanced Accuracy (Validation)')
    plt.xlabel('Model')
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    best_model = df_optimization.iloc[0]['Model']
    best_score = df_optimization.iloc[0]['Validation Score']
    print(f"\n🏆 Best Model: {best_model} (Validation Score: {best_score:.4f})")
    
else:
    print(f"❌ Optimization results not found at {param_file}")
    print("Run the optimization step first!")

## Summary

**✅ Optimization completed successfully!**

**Key improvements in this version:**
- **Faster:** Always uses balanced class weights (no optimization needed)
- **RAM efficient:** Aggressive memory cleanup after each model
- **Cleaner:** Separated optimization from evaluation for better organization

**Files created:**
- `optimal_parameters_normal_latest.json` - Best parameters for each model
- Timestamped backup versions
- Results backed up to your Google Drive

**Next steps:**
1. Use the evaluation notebook to test these models
2. Review best parameters for production use
3. Try different subjects or parameter ranges if needed

**Best performing model:** Check the results above for the top performer!